In [ ]:
!pip install transformers sentencepiece

# ChatbotData 읽기

In [ ]:
!pip install transformers datasets sentencepiece

In [ ]:
import pandas as pd

# CSV 직접 읽기
df = pd.read_csv("/content/ChatbotData.csv")

# 필요한 데이터 전처리
train_data = [f"사용자: {q}\n챗봇: {a}" for q, a in zip(df["Q"], df["A"])]

with open("train.txt", "w") as f:
    for line in train_data:
        f.write(line + "\n")

In [ ]:
df.head(3)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.current_device())

# KoGPT2

In [ ]:
# 1. 토크나이저 및 모델 로딩
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "skt/kogpt2-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))

# 2. ChatbotData.csv pandas로 읽기
import pandas as pd
df = pd.read_csv("/content/ChatbotData.csv")
train_data = [f"사용자: {q}\n챗봇: {a}" for q, a in zip(df["Q"], df["A"])]
with open("train.txt", "w") as f:
    for line in train_data:
        f.write(line + "\n")

# 3. TextDataset/Collator 등 이어서
from transformers import TextDataset, DataCollatorForLanguageModeling, Trainer, TrainingArguments

def get_dataset(file_path, tokenizer, block_size=128):
    return TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=block_size,
        overwrite_cache=True
    )

train_dataset = get_dataset("train.txt", tokenizer)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./kogpt2-chatbot",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=5000,
    save_total_limit=2,
    prediction_loss_only=True,
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset
)

# 4. 모델 학습
trainer.train()
trainer.save_model("./kogpt2-chatbot")
tokenizer.save_pretrained("./kogpt2-chatbot")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("./kogpt2-chatbot")
model = AutoModelForCausalLM.from_pretrained("./kogpt2-chatbot")
model.eval()
model.to("cuda")

def chatbot_response(user_input, max_length=60, top_p=0.92, temperature=0.7):
    input_text = f"사용자: {user_input}\n챗봇:"
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(model.device)
    output = model.generate(
        input_ids,
        max_length=max_length,
        do_sample=True,
        top_p=top_p,
        temperature=temperature,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    # "챗봇:" 이후 ~ "사용자:" 바로 앞까지만 출력 (or 문장 끝까지)
    answer = result.split("챗봇:")[-1].strip()
    # 사용자:가 다시 나오면 그 전까지만
    if "사용자:" in answer:
        answer = answer.split("사용자:")[0].strip()
    return answer

# ---- 실제 대화 루프 -----
print("챗봇에게 물어보세요! (종료하려면 'exit' 입력)")
while True:
    user_input = input("나: ")
    if user_input.lower() in ["exit", "quit", "종료"]:
        print("대화를 종료합니다.")
        break
    answer = chatbot_response(user_input)
    print("챗봇:", answer)

# SNS 데이터 사용해서 학습

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 데이터 압축 해제

In [ ]:
import os
import zipfile

def unzip_all_in_folder(zip_dir, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    zip_files = [os.path.join(zip_dir, f) for f in os.listdir(zip_dir) if f.endswith('.zip')]
    for zf in zip_files:
        with zipfile.ZipFile(zf, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
    print(f"압축 해제 완료: {extract_to}")

# 경로 맞게 수정
train_zip_dir = "/content/drive/MyDrive/통인pbl/chatbot/SNS데이터/Training/라벨링데이터"
val_zip_dir = "/content/drive/MyDrive/통인pbl/chatbot/SNS데이터/Validation/라벨링데이터"

train_extract_dir = "/content/train_json"
val_extract_dir = "/content/val_json"

unzip_all_in_folder(train_zip_dir, train_extract_dir)
unzip_all_in_folder(val_zip_dir, val_extract_dir)

## 멀티턴 대화 프롬프트 추출

In [ ]:
import glob
import json
from tqdm import tqdm  # 진행상황 바

def collect_multiturn_dialogs(json_dir, max_files=None):
    json_files = glob.glob(os.path.join(json_dir, "*.json"))
    if max_files:
        json_files = json_files[:max_files]
    dialogs = []
    for json_file in tqdm(json_files, desc=f"Processing {json_dir}", unit="json"):
        with open(json_file, encoding="utf-8") as f:
            data = json.load(f)
            utterances = data["utterances"]
            dialog = ""
            for utt in utterances:
                if utt["speaker"] == "speakerA":
                    dialog += f"사용자: {utt['text']}\n"
                elif utt["speaker"] == "speakerB":
                    dialog += f"챗봇: {utt['text']}\n"
            if dialog.strip():
                dialogs.append(dialog.strip())
    print(f"{json_dir} 전체 파일 처리 완료 (총 {len(dialogs)}개 대화)")
    return dialogs

train_dialogs = collect_multiturn_dialogs(train_extract_dir)
val_dialogs = collect_multiturn_dialogs(val_extract_dir)

with open("/content/train_multiturn.txt", "w", encoding="utf-8") as f:
    for d in tqdm(train_dialogs, desc="Saving train", unit="dialog"):
        f.write(d + "\n\n")
with open("/content/val_multiturn.txt", "w", encoding="utf-8") as f:
    for d in tqdm(val_dialogs, desc="Saving valid", unit="dialog"):
        f.write(d + "\n\n")

## 토크나이저 및 모델 로딩

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
print("모델과 토크나이저 로딩 중...")
model_name = "skt/kogpt2-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print("모델 및 토크나이저 로딩 완료")

if tokenizer.pad_token is None:
    print("pad_token 추가 중...")
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
    print("pad_token 추가 완료")

## 데이터셋 불러오기

In [ ]:
from transformers import TextDataset, DataCollatorForLanguageModeling

def get_dataset(file_path, tokenizer, block_size=128):
    print(f"데이터셋 로딩 중: {file_path}")
    dataset = TextDataset(
        tokenizer=tokenizer,
        file_path=file_path,
        block_size=block_size,
        overwrite_cache=True
    )
    print(f"데이터셋 로딩 완료: {file_path}, 총 {len(dataset)}개 블록")
    return dataset

train_dataset = get_dataset("/content/train_multiturn.txt", tokenizer)
eval_dataset = get_dataset("/content/val_multiturn.txt", tokenizer)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Trainer 세팅 및 학습

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/kogpt2-sns-chatbot",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    save_steps=2000,
    save_total_limit=2,
    prediction_loss_only=True,
    logging_steps=100,
    report_to="none"
)

print("Trainer 생성 중...")
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
print("Trainer 준비 완료")

print("모델 학습 시작!")
trainer.train()
print("모델 학습 완료. 모델 저장 중...")
trainer.save_model("/content/kogpt2-sns-chatbot")
tokenizer.save_pretrained("/content/kogpt2-sns-chatbot")
print("모델, 토크나이저 저장 완료")

## 테스트

In [ ]:
print("모델 로드 중...")
model = AutoModelForCausalLM.from_pretrained("/content/kogpt2-sns-chatbot")
tokenizer = AutoTokenizer.from_pretrained("/content/kogpt2-sns-chatbot")
model.eval()
model.to("cuda")  # Colab GPU라면

def chat_multiturn(prompt, max_length=60, top_p=0.92, temperature=0.7):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        input_ids,
        max_length=input_ids.shape[1]+30,  # 30 token만 추가 생성(한 턴 정도)
        do_sample=True,
        top_p=top_p,
        temperature=temperature,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = result[len(prompt):].strip()
    # "사용자:" 또는 "챗봇:"이 다시 나오면 그 앞까지만
    for stopword in ["사용자:", "챗봇:"]:
        if stopword in answer:
            answer = answer.split(stopword)[0].strip()
    # 한 줄(엔터) 기준 앞부분만 반환
    answer = answer.split('\n')[0].strip()
    return answer

print("챗봇 대화 모드 시작!")
history = ""

while True:
    user = input("나: ")
    if user.strip().lower() in ["exit", "quit", "종료"]:
        print("대화를 종료합니다.")
        break
    history += f"사용자: {user}\n챗봇:"
    answer = chat_multiturn(history)
    print("챗봇:", answer)
    history += f"{answer}\n"

# KoGPT-2 SNS데이터 학습 모델 저장

In [ ]:
# 모델 폴더를 구글 드라이브에 백업
!cp -r /content/kogpt2-sns-chatbot /content/drive/MyDrive/통인pbl/chatbot/

## 수호가 잠깐 쓸게

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers
!pip install evaluate
!pip install sentencepiece

import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel
import pandas as pd
import evaluate
from tqdm import tqdm

In [ ]:
import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel
import pandas as pd
import evaluate
from tqdm import tqdm

In [ ]:
# 저장된 모델 경로
MODEL_PATH = "/content/drive/MyDrive/통인pbl/chatbot/kogpt2-sns-chatbot"

tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_PATH)
model = GPT2LMHeadModel.from_pretrained(MODEL_PATH).to("cuda")
model.eval()

In [ ]:
def kogpt2_bot(query: str, max_len=64) -> str:
    input_ids = tokenizer.encode(query, return_tensors='pt').to("cuda")
    with torch.no_grad():
        gen_ids = model.generate(input_ids, max_length=max_len, pad_token_id=tokenizer.pad_token_id)
    output = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
    return output[len(query):].strip()  # 질문 이후만 추출

In [ ]:
# 5. CSV에서 평가 데이터 로드
df = pd.read_csv("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/eval_set.csv")  # eval_set.csv 파일이 같은 디렉토리에 있어야 함
eval_questions = df["query"].tolist()
eval_answers = df["response"].tolist()

In [ ]:
import torch
if torch.cuda.is_available():
    print("✅ GPU 사용 중:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU 사용 불가. CPU 사용 중")

In [ ]:
# 6. 챗봇 응답 생성
kogpt2_preds = [kogpt2_bot(q) for q in tqdm(eval_questions)]
model.to("cuda")

In [ ]:
pip install rouge_score

In [ ]:
# 7. 평가 지표 계산
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

results = {
    "BLEU": bleu.compute(predictions=kogpt2_preds, references=eval_answers),
    "ROUGE": rouge.compute(predictions=kogpt2_preds, references=eval_answers),
    "METEOR": meteor.compute(predictions=kogpt2_preds, references=eval_answers)
}

In [ ]:
# 8. 출력
for metric, score in results.items():
    print(f"{metric}:\n{score}\n")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print("모델 로드 중...")
model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/통인pbl/chatbot/kogpt2-sns-chatbot")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/통인pbl/chatbot/kogpt2-sns-chatbot")
model.eval()
model.to("cuda")  # Colab GPU라면

def chat_multiturn(prompt, max_length=60, top_p=0.92, temperature=0.7):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        input_ids,
        max_length=input_ids.shape[1]+30,  # 30 token만 추가 생성(한 턴 정도)
        do_sample=True,
        top_p=top_p,
        temperature=temperature,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = result[len(prompt):].strip()
    # "사용자:" 또는 "챗봇:"이 다시 나오면 그 앞까지만
    for stopword in ["사용자:", "챗봇:"]:
        if stopword in answer:
            answer = answer.split(stopword)[0].strip()
    # 한 줄(엔터) 기준 앞부분만 반환
    answer = answer.split('\n')[0].strip()
    return answer

print("챗봇 대화 모드 시작!")
history = ""

while True:
    user = input("ME: ")
    if user.strip().lower() in ["exit", "quit", "종료"]:
        print("대화를 종료합니다.")
        break
    history += f"🙋‍♀️ ME: {user}\n챗봇:"
    answer = chat_multiturn(history)
    print("🤖 챗봇:", answer)
    history += f"{answer}\n"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("/content/drive/MyDrive/통인pbl/chatbot/kogpt2-sns-chatbot")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/통인pbl/chatbot/kogpt2-sns-chatbot")
model.eval()
model.to("cuda")

In [ ]:
def chat_multiturn(prompt, max_length=60, top_p=0.92, temperature=0.7):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        input_ids,
        max_length=input_ids.shape[1]+30,
        do_sample=True,
        top_p=top_p,
        temperature=temperature,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = result[len(prompt):].strip()
    for stopword in ["사용자:", "챗봇:"]:
        if stopword in answer:
            answer = answer.split(stopword)[0].strip()
    answer = answer.split('\n')[0].strip()
    return answer

In [ ]:
def kogpt_bot(question: str) -> str:
    prompt = f"사용자: {question}\n챗봇:"
    return chat_multiturn(prompt)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

# 평가셋 로딩
eval_df = pd.read_csv("/content/drive/MyDrive/통인pbl/chatbot/Korean_Data/eval_set.csv")
eval_questions = eval_df["query"].tolist()
eval_answers   = eval_df["response"].tolist()

# 예측 수집
kogpt_preds = [kogpt_bot(q) for q in eval_questions]

# BLEU 계산
def compute_bleu(pred, ref):
    return sentence_bleu([ref.split()], pred.split(), weights=(0.5, 0.5))

bleu_kogpt = sum(compute_bleu(p, r) for p, r in zip(kogpt_preds, eval_answers)) / len(eval_answers)

print(f"🔹 KoGPT 평균 BLEU: {bleu_kogpt:.4f}")

# BERTScore 계산
P_k, R_k, F1_k = score(kogpt_preds, eval_answers, lang="ko", verbose=True)

print(f"🟢 KoGPT BERTScore F1: {F1_k.mean().item():.4f}")